
# Project phases 

## Fase 0 — Fundações

- Definição do escopo físico e dicionário de variáveis
- Contratos de IO entre Módulo 1 e Módulo 2 (unidades, formato xarray/NetCDF)
- Scaffold do repositório (configs, CI, Docker, ambiente Conda)
- Definição dos solvers de referência por sub-problema
- Plano de amostragem LHS no espaço paramétrico

## Fase 1 — Neural Operator replacement for a multibody coupled PRW-WSI system

This phase develops a Physics Informed Neural Operator (PINO) framework that replaces Phase Resolving Wave (PRW) models and Wave-Structure Interaction (WSI) solvers built on BEM (Boundary Element Method) in the simulation of multibody arrays, enabling fast prediction of wave fields, wave-wave interactions, floating object response, and in case o WECs, the farm performance without numerical solvers at runtime. The learned system predicts both the incoming and scattered wave field and the array scale hydrodynamic response of multiple bodies without calling a PRW or BEM solvers during inference.

It must be a full replacement program for the two most expensive layers in wave farm simulation with the end goal of producing a learned solver that is fast enough for layout optimization, control studies, large scenario sweeps, and real-time monitoring while remaining faithful to the governing physics and to trusted numerical references. 

Phase 1 itself becomes the moment where numerical solvers stop being the runtime engine and become only the source of training and validation data. In other words, OceanWave3D, SWASH, FUNWAVE, Capytaine, HAMS, NEMOH, and similar tools remain the reference standard for generating training data, defining benchmark cases, and auditing extrapolation, but are not operational tools. For numerical solvers, more complete physics means more runtime cost; for a trained neural operator, more complete physics mainly means more careful training design, not more runtime cost in the same explosive way.

The PRW module, receive external inputs (initial boundary conditions) computes the local wave field over bathymetry and around obstacles or devices. The WSI  module, which computes excitation, radiation, diffraction, body motions, PTO relevant response, and power metrics for one device or an interacting array. 

> Phase 1 objective: 
1. Develop a physics informed operator learning framework that replaces PRW models and BEM based WSI models, using numerical solvers only for offline dataset generation, calibration, and benchmark validation.

2. The learned system must replace the numerical solvers in runtime evaluation for new sea states, layouts, locations, and device parameters in a wide validity envelope. To be used in rapid screening, design iterations, and control aware and operational forecasting workflows with data assimilation and uncertainty parameters.

### F1A - Module A: Array WSI operator.

Maps local wave field, device properties, and array configuration to excitation loads, hydrodynamic coefficients, body response, absorbed power, and interaction feedback terms. This is the learned replacement for BEM based tools such as Capytaine, HAMS, or NEMOH for the selected class of devices and hydrodynamic assumptions.
- **F1A1** — Resolver WSI  obstáculo flutuante com e sem damping
- Dados de datasets opensource, modelos numéricos, físicos e analíticos
- Entradas: raio, draft, massa, Bpto, profundidade
- Saídas: A(ω), B(ω), Fex(ω), RAO, potência absorvida
- Arquitetura: **DeepONet parametrizado**
- Validação cruzada contra Capytaine/HAMS em casos não vistos
- **Paper:** WSI-BEM com DeepONet
  
* **F1A2** — Número variável de dispositivos com interação hidrodinâmica entre pares
* Arquitetura: **Graph Neural Operator** (WEC = nó, interação = aresta)
* Linha de controlo paralela: superposição linear como baseline físico
* É aqui que entra o `MultiObjectFSI` 
* **Paper:** GNO multi-WEC para fazendas de geometria arbitrária

### F1B - Module B: Phase resolving wave field operator.

Maps bathymetry, incident wave conditions, wave-wave interaction, and boundary conditions to the local phase resolved wave field and near device kinematics over the domain. This is the learned replacement for tools such as OceanWave3D, SWASH, or FUNWAVE within the selected regime.

- **F1B1** — propagação pura sem WECs, generalização entre batimetrias
- **F1B2** — obstáculos simplificados fixos, presença de corpos como termos equivalentes
- **F1B3** — operador aceita layout explícito + termos vindos do Módulo Onda
- Dados: Modelos numéricos e analíticos, boias, tanque ondas,  etc
- Arquitetura: **GINO** ou **WNO**
- Saídas: η(x,y,t), velocidades, pressões nos pontos de interesse
- **Paper:** operador de propagação generalizado

### F1C -  Bidirectional coupling F1A-F1B

Combines Modules A and B so domains that include obstacles or devices arrays can be evaluated end to end, with variable number of bodies, interaction between them, and array level performance outputs in case of WECs.

- Iterativo primeiro: F1B → campo incidente → F1A → forças de radiação → volta ao F1B
- Depois: acoplamento end-to-end com treino conjunto
- Validação contra dados modelos fisicos, numéricos e de operação
- **Paper:** framework acoplado onda-WEC

### F1D (opcional) — Otimização de Layout

- Acoplado integrado com **Algoritmo Genético**
- Substituição direta do SWAN/SWASH no loop de otimização
- Objetivos: potência, robustez direcional, carregamentos, separação mínima
- Caminho futuro: Bayesian optimization, multi-objetivo com gradiente
- **Paper:** GA + neural operator para otimização de fazendas

### F1E (optional) — Digital Twin Operacional

- Assimilação de dados reais: boias, ADCP, HF Radar, satélite
- 4DVar-PINO: estado otimizado com observações esparsas
- Parâmetros físicos aprendíveis (νh, νv, Manning) em vez de constantes fixas
- Quantificação de incerteza: ensemble + conformal prediction
- Dashboard FastAPI em tempo real



## Scientific modules

The scientific modules should be defined as first-class subsystems rather than utility functions. The **wetting and drying module** handles regularized water column thickness, wet masks, and dry-cell velocity suppression. The **atmospheric forcing module** computes or injects air-sea stresses and heat-like surface forcing. The **object interaction module** handles fixed and floating bodies, state updates, body-force spreading, solid masks, and later pairwise structure interaction. Your notes already show why these should sit outside the operator core but be coupled into the forward pass in a disciplined order. 

The coupling order should be formalized early and kept stable. A good Phase 1 statement is: bathymetric and wetting-drying regularization first, external surface forcing second, body interaction third, operator propagation fourth, post-processing masks last. That ordering is consistent with the coupled FullOceanPINO idea you developed and prevents scientific ambiguity about where each physical mechanism enters the state update. 

## Uncertainty architecture

Uncertainty should be part of the architecture from the beginning, but kept modest in Phase 1. Your notes already identify ensembles, MC dropout, probabilistic operators, and conformal calibration as useful strategies, and they correctly link uncertainty to operational credibility and digital twins. The formal architecture should therefore include a dedicated `uncertainty/` package from the first release, even if its initial implementation is simple. 

For Phase 1, the right target is not a fully Bayesian operator stack. It is a practical uncertainty layer with three responsibilities: produce spread estimates, calibrate intervals on held-out data, and export uncertainty maps alongside deterministic predictions. That is enough to make the system useful for engineering triage and later digital twin escalation logic, where low-confidence regions trigger solver fallback, operator rejection, or human review. This directly supports your digital twin goal without blowing up the initial coding scope. 

## Digital twin readiness

Digital twin readiness should be designed structurally, not marketed prematurely. In formal terms, Phase 1 does not need full online assimilation, live sensor ingestion, or state estimation loops. What it does need is an architecture that can host those later without being rewritten. Your notes already point in this direction when they discuss modular scientific software, observation interfaces, and later uncertainty-aware operational use. 

That means the codebase should already include standard dataset contracts, metadata-rich outputs, a place for observation operators, and interfaces that make future assimilation possible. In other words, Phase 1 should be digital twin compatible, not yet a finished digital twin product. That distinction matters because it keeps the research program honest. 

## Repository architecture

The repository should follow a scientific software layout, not a notebook-first layout. Your earlier planning repeatedly argued for xarray or NetCDF-friendly datasets, modular packages, YAML-driven experiments, tests, logging, and reproducibility. That is the correct foundation. 

A formal top-level structure for the real project should look like this in substance: `src/nossomar/` for code, `configs/` for experiments and domains, `scripts/` for entry points, `tests/` for unit and integration tests, `docs/` for architecture and phase documents, `data/` for manifests and templates rather than raw heavy data, and `notebooks/` only for bounded reproducible analysis. This is closely aligned with the good parts of your scaffold notes, while avoiding the trap of giving equal weight to too many speculative modules too early. 

## The data strategy

 A neural operator trained with full Navier–Stokes based physics is not automatically “better” than reduced models everywhere. It is better only if the data, resolution, training curriculum, and validation are aligned with the target regime. Your notes already show that bad temporal resolution, coarse spatial resolution, or training on datasets with no phase information would break the promise immediately.

The datasets must be from both solver families. Numerical references: OceanWave3D or equivalent for PWR, and Capytaine, HAMS, and WEC-sim for WSI. The training data should have two streams. One stream for wave field solutions over parametrized domains, bathymetries, and incident conditions. The other stream for WSI solutions over parametrized devices, frequencies, headings, and array layouts. These streams must then be linked through a common representation of local wave conditions at each device and a consistent data contract for geometry, coordinates, and metadata. Your earlier notes about freezing IO conventions were exactly right.

If use a full physics informed operator built around oceanic Navier–Stokes in sigma coordinates, the project does not need to switch scientific identity every time the depth regime changes. Instead, one operator family can learn across regimes, provided the training data and configuration support that scope. That is a big conceptual advantage for a platform like NOSSO-MAR, because it supports long term aim of a single computational layer between trusted numerical solvers and real applications in design, optimization, forecasting, and decision support.

### Data source hierarchy (lighter to hevier)

**Nível 0 — Soluções analíticas (custo zero)**
Potencial de velocidade para cilindro, Teoria Linear de Ondas, solução de Airy, Green analítico para geometrias simples. Geram milhares de casos em segundos sem nenhum solver.

**Nível 1 — Datasets públicos já existentes**
CMEMS (oceano), WEC-Sim benchmark cases (NREL GitHub), ETOPO batimetria, ERA5 forçantes atmosféricas. Já estão gerados, são gratuitos e cobrem condições realistas.

**Nível 2 — Capytaine em modo leve**
Só para validação e casos fora da cobertura analítica. 500–2000 casos. Com LHS focado nas zonas de maior gradiente de A(ω).

Capytaine is the best development backbone because it has a native Python API and integrates naturally with xarray and ML workflows, even if HAMS may later become attractive for mass data generation speed.Capytaine first for development, automation, and dataset schema design; HAMS or HAMS MREL later if need faster bulk generation for larger array datasets.

**Nível 3 — OceanWave3D / OpenFOAM**
Só para os casos de validação de alta fidelidade do paper. Nunca para treino em massa.


# Plano de Implementação Fase 0 e 1


The development plan should be framed as a research program with a deliberately narrow executable baseline. Phase 0 is architecture hardening and interface freezing. Phase 1 is the first end-to-end scientific surrogate. Later phases extend physics, geometry, coupling sophistication, and operationality. 

Phase 1 should target a concrete canonical problem such as near-field phase-resolving wave propagation with one or a few idealized WEC or obstacle configurations, fixed bathymetry class, controlled forcing, and solver-derived training data. Propose a baseline of this kind, for example a cylindrical or simple-body WEC case with a complete solver-to-loss pipeline and reproducible reference experiment. 

This is the right choice because it keeps the scope sharp. It allows test operator learning, physical residuals, object coupling hooks, and uncertainty calibration without yet pretending to model every relevant marine process. It also produces something publishable and extensible. 

### Coding order
The coding order should be disciplined. First freeze `field_schema.py`, `types.py`, and the Phase 1 config contract. Second build the data pipeline. Third implement the operator baseline. Fourth implement losses. Fifth make the trainer run end to end. Sixth add uncertainty. Seventh add evaluation and benchmark scripts. Only after that should you deepen coupling complexity. This order mirrors the strongest advice in your notes: freeze interfaces, implement the smallest real scientific baseline, and only then expand. 

Phase 1 should privilege the files that are necessary for the first benchmark to run cleanly. Everything else should either be postponed or marked clearly as extension scaffolding. That is how you avoid repeating the earlier “beautiful scaffold, incomplete project” problem already documented in your notes. 



## Fase 0 — Task por Task
> **Para workers agentic:** REQUIRED SUB-SKILL: `superpowers:subagent-driven-development` por task. Steps usam checkbox `- [ ]` para tracking.

**Goal:** Construir a base estrutural para os arquivos do projeto NOSSO-MAR fase a fase, com o mínimo possível de dados de treino vindos de solvers numéricos — usando física analítica, dados públicos e geração sintética inteligente.

**Architecture:** Módulos independentes com contratos IO definidos, treinados progressivamente. Dados de treino gerados por física analítica (soluções fechadas) e datasets públicos (CMEMS, WEC-Sim benchmarks) antes de recorrer a solvers pesados. Cada módulo produz um artefacto testável e publicável antes de avançar.

**Tech Stack:** PyTorch, neuraloperator, Capytaine, xarray/NetCDF, pytest, Zarr


## Fase 0 
### Task 0 — Contratos IO e Tipos

Fix the state variables, the role of sigma coordinates, the input and output tensor conventions, the forcing interfaces, and the exact meaning of each module. This prevents architecture drift later and matches your own earlier insistence on freezing IO conventions early

**Files:**
- Criar: `src/nossomar/core/contracts.py`
- Criar:
- [ ] **Step 1: Escrever o teste RED**`tests/test_contracts.py`
- [ ] **Step 2: Rodar — deve FALHAR** `pytest tests/test_contracts.py -v`
- [ ] **Step 3: Implementar** src/nossomar/core/contracts.py
- [ ] **Step 4: Rodar** `pytest tests/test_contracts.py -v`
- [ ] **Step 5: Commit** `git commit -m "feat: IO contracts WECState + WaveField"`

## Phase 1A
### Task 1 — Dados Analíticos Fase 1 (Nível 0)
**Files:**`src/nossomar/data/analytic_wec.py`
- [ ] **Step 1: Escrever o testle RED**`tests/test_analytic_wec.py`
- [ ] **Step 2: Rodar — deve FALHAR**
- [ ] **Step 3: Implementar com teoria linear + solução de Hulme (1982)** `src/nossomar/data/analytic_wec.py`
- [ ] **Step 4: Rodar — deve PASSAR** `pytest tests/test_analytic_wec.py -v` → 2/2 pass
- [ ] **Step 5: Commit** `git commit -m "feat: analytic BEM surrogates for cylinder WEC"`

### Task 2 — Dataset Pipeline (xarray → Zarr)
**Files:**`src/nossomar/data/wec_dataset.py`
- [ ] **Step 1: Teste RED** `tests/test_wec_dataset.py`
- [ ] **Step 2: FALHA**
- [ ] **Step 3: Implementar** — LHS em (radius, draft, depth, Bpto) → Capytaine quando disponível, analítico quando não
- [ ] **Step 4: PASSA** `pytest tests/test_wec_dataset.py -v`
- [ ] **Step 5: Commit**

### Task 3 — DeepONet Módulo F1A
**Files:**`src/nossomar/operators/deeponet_wec.py`
- [ ] **Step 1: Teste RED — forward pass shape** `tests/test_deeponet_wec.py`
- [ ] **Step 2: FALHA**
- [ ] **Step 3: Implementar**
- [ ] **Step 4: PASSA**
- [ ] **Step 5: Commit**

### Task 4 — Training Loop Fase 1

**Files:**`src/nossomar/training/train_wec.py`
- [ ] **Step 1: Teste RED — loss desce em 10 epochs**  `tests/test_train_wec.py`
- [ ] **Step 2: FALHA**
- [ ] **Step 3: Implementar loop com PCGrad opcional**
- [ ] **Step 4: PASSA** — 2/2 assertions
- [ ] **Step 5: Commit**

### Task 5 — Validação com Dados Públicos (NREL WEC-Sim)
**Files:**
`src/nossomar/data/public_benchmarks.py`
`scripts/validate_phase1.py`
- [ ] **Step 1: Download e parse do benchmark cylinder NREL**
- [ ] **Step 2: Teste RED — erro relativo < 15% em A(ω) para RM3**
- [ ] **Step 3: Escrever e Implementar loader RM3**
    - [ ] Normaliza outputs em:`omega`, `added_mass_heave`, `draft`, `depth`, `bpto`, `radius`.
- [ ] **Step 4: Add script de validação e execução 
- [ ] **Step 4: PASSA ou ajusta threshold com base no erro real**
- [ ] **Step 5: Commit** — este resultado é a métrica do paper Fase 1

#### The full scientific benchmark
For Phase 1, the scientifically defensible target is a **public heave added-mass benchmark** for a canonical body, with frequency-resolved data and known geometry. A proper benchmark for this phase should do four things:

1. load real benchmark data
2. normalize geometry and frequency axes
3. compute model predictions on the same grid
4. report RMSE and relative RMSE, ideally around resonant and off-resonant regions separately too.

### Task 6 — Integração MultiObjectFSI no Loop
**Files:**
- Modificar: `src/nossomar/coupling/coupled_pino.py`
- Criar: `tests/test_multi_object_fsi.py`
- [ ] **Step 1: Teste RED — N=3 WECs produz output sem NaN**
- [ ] **Step 2: FALHA** (contratos novos)
- [ ] **Step 3: Adaptar MultiObjectFSI para usar WECState/WaveField**
- [ ] **Step 4: PASSA**
- [ ] **Step 5: Commit**

### Dispatch Order

After the implementer replies DONE, dispatch a **spec reviewer** (did the code match what the task asked, nothing more, nothing less), then a **code quality reviewer** (clean code, no magic numbers, types correct). Only then mark Task 1 complete and move to Task 2. 

These are independent enough to go one-at-a-time with no parallel conflicts:

| \#  | Task             | Key deliverable                               | Complexity signal                               |
| :-- | :--------------- | :-------------------------------------------- | :---------------------------------------------- |
| 0   | Contracts        | `WECState`, `WaveField`, `validate_wec_state` | Cheap model — 2 files, clear spec               |
| 1   | Analytic WEC     | `analytic_wec.py` — A(ω), B(ω) for cylinder   | Cheap model — pure math, no deps                |
| 2   | Dataset pipeline | `wec_dataset.py` — LHS → xarray → Zarr        | Cheap model — 1 file, standard libs             |
| 3   | DeepONet         | `deeponet_wec.py` — branch+trunk net          | Standard model — multi-layer NN design          |
| 4   | Training loop    | `train_wec.py` — AdamW + cosine schedule      | Standard model — integrates Tasks 2–4           |
| 5   | Validation       | `validate_phase1.py` — benchmark runner       | Standard model — integrates all above           |
| 6   | Multi-object FSI | `multi_object_fsi.py` — GAT + GRU + spread    | **Capable model** — multi-file, judgment needed |

Só depois de Task 5 passar com erro < 15% é que faz sentido gerar dados. Isso evita gastar semanas em dados de treino que podem não ser necessários.

The `finishing-a-development-branch` skill kicks in:

1. Run `pytest tests/` — all must pass before anything else
2. Choose one of 4 options: merge locally / push + PR / keep as-is / discard
3. Clean up worktree



## Phase 1 deliverables and coding roadmap

Phase 1 should deliver a working package, not just documents and stubs. The required deliverables are: a reproducible dataset pipeline, a trainable FullOceanPINO baseline, a minimal but real coupling interface, an uncertainty API, evaluation reports against solver data, and tests that prove the pipeline actually runs. Your notes explicitly criticize prior scaffold generations for stopping at folder trees and placeholders, and they argue that a real Phase 1 only exists once a minimal training experiment runs end to end and shows loss descent with benchmark outputs. 

That means the coding roadmap must favor executable substance over breadth. If a module is not required to make the Phase 1 experiment train, evaluate, and compare, it should either be a clearly marked stub or be postponed. 

Below is the file-by-file roadmap I would use for Phase 1. The order reflects implementation dependencies and the logic already present in your notes about freezing IO contracts, building the operator core, then adding training and benchmark structure. 

### `README.md`

This file should define Phase 1 precisely: scientific objective, current scope, what is implemented, what is explicitly not implemented yet, how to install, how to run training, how to run evaluation, and what reference benchmark is considered the baseline. Your scaffold discussions show that the README matters because collaborators need one source of truth, but it should stay neutral and scientific rather than locking the whole project identity too early into optional later layers like MARL. 

### `docs/architecture.md`

This is the formal architecture document. It should state the system layers, scientific assumptions, variable conventions, coupling order, uncertainty strategy, and extension path to later phases. This file is where you place the exact argument for keeping full governing physics as the structural prior while moving runtime cost into training and validation. That framing comes straight from your FullOceanPINO planning logic. 

### `docs/phase1.md`

This file should describe the Phase 1 experiment as a contract: target PDE scope, geometry assumptions, inputs, outputs, loss terms, evaluation metrics, uncertainty target, and success criteria. Your notes already recommend crystallizing a fixed reference experiment with solver-vs-surrogate plots and fixed hyperparameters. This document is where that becomes formal. 

### `configs/phase1/base.yaml`

This is the global experiment config. It should include grid resolution, time window lengths, variable names, normalization choices, model width and depth, training hyperparameters, loss weights, and uncertainty settings. Your planning notes repeatedly emphasize config-driven reproducibility and frozen conventions. 

### `configs/phase1/domain.yaml`

This file should define the physical domain assumptions for the canonical case: extent, depth model, boundary condition type, object layout schema, and forcing options. Keeping domain assumptions separate from training hyperparameters makes later scaling cleaner and matches the modular design you have been building toward. 

### `src/nossomar/__init__.py`

Keep this minimal and stable. Version export, package identity, and nothing fancy. Phase 1 benefits from boring package boundaries. 

### `src/nossomar/core/field_schema.py`

This is one of the most important files in the whole project. It should define the canonical tensor and metadata contracts: variable ordering, dimensions, coordinate conventions, units, masks, static fields, and optional forcings. Your notes already stress that freezing IO contracts early is the next logical step after scaffolding. Without this file, the codebase will drift. 

### `src/nossomar/core/types.py`

Use this for typed dataclasses or typed dictionaries representing model inputs, outputs, uncertainty outputs, object states, and evaluation summaries. The goal is not ceremony. It is to make interfaces explicit and stop silent shape drift. That fits the scientific-software emphasis in your planning material. 

### `src/nossomar/data/io.py`

This file should load and save xarray or NetCDF-style datasets, convert them into model-ready tensors, and preserve metadata where possible. Your notes strongly support xarray and NetCDF as technically correct choices for oceanographic fields and reproducible scientific pipelines. 

### `src/nossomar/data/preprocess.py`

This file should implement normalization, temporal window extraction, static-field preparation, mask generation, and train-validation-test splitting helpers. It must be deterministic and config-driven. Phase 1 needs this because benchmark credibility depends on reproducible data preparation rather than ad hoc notebook preprocessing. 

### `src/nossomar/data/dataset.py`

This should expose the PyTorch dataset classes for Phase 1. It must return structured batches with dynamic state, static descriptors like bathymetry, optional forcings, and target fields. Your earlier notes make clear that the project only becomes real when the solver-to-loader-to-loss chain is complete. 

### `src/nossomar/operators/spectral.py`

This file should implement the spectral convolution machinery and differential helpers used by FullOceanPINO. It belongs here rather than hidden inside the model file because it is reusable and scientifically central. Your FullOceanPINO notes already assume this sort of decomposition. 

### `src/nossomar/operators/conditioning.py`

Use this for bathymetry and static-field encoders, FiLM-like modulation, or static conditioning blocks. Your notes explicitly mention bathymetry conditioning and also warn that bad bathymetric handling can collapse the model’s ability to distinguish shelf from deeper regions. That makes this file scientifically important, not cosmetic. 

### `src/nossomar/operators/blocks.py`

This should hold reusable operator blocks such as space-time residual blocks, spectral residual units, and projection layers. It keeps the main model readable and testable. The FullOceanPINO concept in your notes is already modular enough that this split makes sense. 

### `src/nossomar/operators/full_ocean_pino.py`

This is the heart of Phase 1. It should contain the FullOceanPINO baseline with a clean forward interface, support for static conditioning, and hooks for coupling modules. If you decide to keep the Phase 1 core slightly narrower than the full long-term vision, this file can still be architected in the long-term style while exposing only the subset required by the baseline benchmark. That would stay faithful to your current plan while preserving future growth. 

### `src/nossomar/physics/differential.py`

This file should hold derivative utilities used by losses and diagnostics. It should be separate from operator internals so that physics losses do not depend on model implementation details. This separation is part of building a scientific platform rather than a single monolithic model. 

### `src/nossomar/physics/loss_full_ns.py`

This file should implement the full-physics residual loss used in training, or the Phase 1 subset of it if you narrow the first milestone. Your notes repeatedly emphasize that the project’s strength comes from retaining governing physics rather than reducing to a black-box predictor. This file is where that claim becomes code. 

### `src/nossomar/modules/wetting_drying.py`

Phase 1 should include this file even if it is initially simple, because your notes make a strong case that coastal and nearshore applications cannot ignore wetting and drying when inundation or shallow transitions matter. The initial implementation can be regularized and limited, but the interface should already exist. 

### `src/nossomar/modules/atmospheric_forcing.py`

If atmospheric forcing is part of the canonical Phase 1 case, implement a minimal real version. If it is not yet active, keep it as a disciplined stub with a fixed interface and tests. Your notes are clear that this process matters scientifically, but Phase 1 should avoid exploding scope if the baseline benchmark does not truly require it. 

### `src/nossomar/modules/object_interaction.py`

This file should define the Phase 1 object interaction interface. For Phase 1, I would recommend a clean abstraction that supports fixed and simple floating bodies with a limited dynamics model, while leaving more complex multi-object interaction to a later extension. Your notes show both the importance of object interaction and the danger of pretending single-object logic solves multi-object hydrodynamic interaction. This file should therefore be designed with extensibility in mind from day one. 

### `src/nossomar/modules/multi_object_interaction.py`

I would include this file in Phase 1 only if you want the architecture to acknowledge the limitation formally. It can start as a placeholder interface plus tests and a clear `NotImplementedYet` path, or as a minimal pairwise interaction correction block. That would directly reflect your own requirement that NOSSO-MAR should not repeat the limitation of a FullOceanPINO coupling layer that only handles isolated floaters. 

### `src/nossomar/coupling/coupled_model.py`

This file should define the ordered coupling logic between wetting and drying, forcing, object interaction, and the operator core. Even if some modules are light in Phase 1, the coupling architecture should already exist. That preserves the long-term identity of the project and matches the sequence you have already described. 

### `src/nossomar/uncertainty/base.py`

Define the uncertainty interface here: mean prediction, spread estimate, calibration state, confidence map, and export schema. Keeping uncertainty behind a standard interface from the beginning is the easiest way to stay digital-twin ready without overengineering the first release. 

### `src/nossomar/uncertainty/ensemble.py`

This file should implement a practical deep ensemble wrapper for Phase 1. Ensembles are simple, robust, and consistent with your notes as an early operational uncertainty tool. 

### `src/nossomar/uncertainty/conformal.py`

This file should implement conformal calibration on held-out validation data. That gives you calibrated intervals even if the base uncertainty estimate is imperfect, and it fits well with your own emphasis on operational trust rather than just raw prediction. 

### `src/nossomar/training/losses.py`

Use this file to assemble supervised loss, physics loss, coupling penalties, regularizers, and optional uncertainty-aware terms. Keeping composition separate from physics implementation makes the trainer simpler and helps later experimentation. 

### `src/nossomar/training/trainer.py`

This should be the real Phase 1 training engine, not a stub. It should load config, build data, instantiate model, compute losses, save checkpoints, log metrics, and support small reference runs. Your notes are very clear that a genuine Phase 1 exists only when a tiny benchmark can actually train and show loss descent. 

### `src/nossomar/training/curriculum.py`

Because your notes discuss warm-up and later curriculum-style introduction of physics, this file should manage loss scheduling and staged training. That is a practical way to keep optimization stable when combining supervised and physics-informed objectives. 

### `src/nossomar/inference/rollout.py`

This file should implement autoregressive rollout, state window updates, optional forcing updates, and uncertainty propagation during prediction. Your notes repeatedly mention long-horizon rollout as a real validation challenge, so this deserves its own tested implementation. 

### `src/nossomar/eval/metrics.py`

This file should define deterministic and probabilistic metrics: RMSE-like errors, spectral consistency checks, structure response metrics if relevant, calibration error, coverage, and sharpness. Phase 1 needs a small but disciplined metric set tied directly to the reference benchmark document. 

### `src/nossomar/eval/benchmark.py`

This should run the fixed reference experiment and produce comparable outputs across model versions. Your notes argue that the project becomes paper-ready only when the reference experiment is crystallized and reproducible. This file is how you encode that. 

### `scripts/train_phase1.py`

This is the main training entry point. It should be boring and reliable: parse config, set seeds, call the trainer, save artifacts. Avoid putting logic here that belongs in the package. 

### `scripts/eval_phase1.py`

This runs benchmark evaluation from saved checkpoints. It should generate metric tables and figures used for internal validation and later paper drafts. 

### `scripts/calibrate_uncertainty.py`

This file should calibrate conformal intervals or uncertainty scaling on validation data. If uncertainty is a first-class architectural promise, it needs an explicit script, not hidden notebook code. 

### `scripts/run_rollout.py`

This should run a deterministic or uncertainty-aware rollout on a selected case. It becomes the first seed of future operational workflows and digital twin style scenario replay. 

### `tests/test_field_schema.py`

This test should verify tensor shapes, variable ordering, and metadata assumptions. This may sound dull, but it prevents a huge amount of scientific and engineering pain later. 

### `tests/test_dataset.py`

This should verify that dataset windows, static fields, and targets are produced consistently and deterministically. A scientific model is only as reproducible as its data pipeline. 

### `tests/test_model_forward.py`

This test should confirm that the Phase 1 model runs a forward pass on a tiny synthetic batch and produces outputs with the right shape and no NaNs. That is the minimum baseline for sanity. 

### `tests/test_physics_loss.py`

This test should validate that the physics loss computes finite values and responds sensibly to known perturbations. Since your claim is that physics remains central, this part needs direct testing. 

### `tests/test_training_smoke.py`

This should run a tiny 1 to 2 epoch training smoke test and verify that the pipeline executes, checkpoints are written, and loss decreases at least modestly. Your notes explicitly say this kind of test is what turns the project from scaffold to working baseline. 

### `tests/test_uncertainty.py`

This should verify ensemble aggregation, interval generation, and basic calibration behavior on toy data. Since uncertainty is part of the architecture, it deserves early tests. 


<REPLACED_CONTENT>


# Test: A wec farm in viana do castelo, offshore of the wind float project.

Ensaio técnico-científico fechado: local conceptual ao largo de **Viana do Castelo**, a oeste do cluster **WindFloat Atlantic**, parque inicial de **point absorbers**, matriz curta de estados de mar, e  hidrodinâmico para resposta WEC e testes de sensibilidade de layout.

## Estrutura de Teste

Eu prepararia estes ficheiros como base:

- `phase1_viana/configs/viana_phase1.yaml`
- `phase1_viana/src/nossomar/site/viana_phase1.py`
- `phase1_viana/scripts/run_viana_phase1.py`
- `phase1_viana/scripts/export_phase1_summary.py`
- `phase1_viana/tests/test_viana_phase1.py`
- `phase1_viana/docs/phase1_plan.md`

O `yaml` define o local conceptual, o parque, os estados de mar e a runtime config. O módulo `viana_phase1.py` constrói o layout 3 por 3, gera as propriedades dos dispositivos e cria um campo de onda sintético de teste. O `run_viana_phase1.py` chama o `CoupledOceanPINO`, usa o estado atual do `WECDeepONet` e do `MultiObjectFSI`, e grava um `json` com métricas do ensaio. O `export_phase1_summary.py` transforma isso num resumo legível em `md`. Os testes confirmam contagem de dispositivos, monotonia da grelha de frequência e consistência das formas tensoriais. 

O documento técnico da Fase 1 tem de dizer claramente: por que Viana, por que um parque 3 por 3, por que point absorbers, quais hipóteses físicas entram, quais saídas são realmente confiáveis nesta fase e o que fica para a Fase 2. E incluir um pequeno anexo com:

- envelope do local
- layout base
- estados de mar usados
- variáveis de entrada do surrogate
- métricas reportadas
- limitações da fase
- ponte para Fase 2 com Capytaine/HAMS multi-corpo e propagação explícita

